In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



In [57]:
df = pd.read_csv("/kaggle/input/datasets/logeshm0324/pjme-dataset/df_for_EDA.csv")

In [58]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,Lag_24,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24
0,1998-12-24 01:00:00,27213.0,1,3,12,24,52,1998,0,28570.0,27669.0,26498.0,29475.375000,30894.589286,2736.646326
1,1998-12-24 02:00:00,25643.0,2,3,12,24,52,1998,0,27213.0,26162.0,25147.0,29453.750000,30897.541667,2765.861628
2,1998-12-24 03:00:00,24907.0,3,3,12,24,52,1998,0,25643.0,25483.0,24574.0,29429.750000,30899.523810,2804.050165
3,1998-12-24 04:00:00,24721.0,4,3,12,24,52,1998,0,24907.0,25045.0,24393.0,29416.250000,30901.476190,2826.766154
4,1998-12-24 05:00:00,25144.0,5,3,12,24,52,1998,0,24721.0,25030.0,24860.0,29421.000000,30903.166667,2819.160745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145193,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,45787.0,42112.0,40343.500000,41856.327381,2295.270146
145194,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,45209.0,40797.0,40282.750000,41873.910714,2177.148998
145195,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,43663.0,38819.0,40230.208333,41895.238095,2106.081917
145196,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,41581.0,36287.0,40171.166667,41918.315476,2086.336995


In [59]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24"
]

target = "PJME_MW"

X = df[features]
y = df[target]

In [60]:
X

,PJME_MW,Hour,Day,Week,Month,DayOfWeek,IsWeekend,Lag_1,Lag_24,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24
0,27213.0,1,24,52,12,3,0,28570.0,27669.0,26498.0,29475.375000,30894.589286,2736.646326
1,25643.0,2,24,52,12,3,0,27213.0,26162.0,25147.0,29453.750000,30897.541667,2765.861628
2,24907.0,3,24,52,12,3,0,25643.0,25483.0,24574.0,29429.750000,30899.523810,2804.050165
3,24721.0,4,24,52,12,3,0,24907.0,25045.0,24393.0,29416.250000,30901.476190,2826.766154
4,25144.0,5,24,52,12,3,0,24721.0,25030.0,24860.0,29421.000000,30903.166667,2819.160745
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145193,44284.0,3,2,9,3,6,1,44343.0,45787.0,42112.0,40343.500000,41856.327381,2295.270146
145194,43751.0,4,2,9,3,6,1,44284.0,45209.0,40797.0,40282.750000,41873.910714,2177.148998
145195,42402.0,5,2,9,3,6,1,43751.0,43663.0,38819.0,40230.208333,41895.238095,2106.081917
145196,40164.0,6,2,9,3,6,1,42402.0,41581.0,36287.0,40171.166667,41918.315476,2086.336995


In [61]:
y

0         27213.0
1         25643.0
2         24907.0
3         24721.0
4         25144.0
           ...   
145193    44284.0
145194    43751.0
145195    42402.0
145196    40164.0
145197    38608.0
Name: PJME_MW, Length: 145198, dtype: float64

In [62]:
train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)

X_train = X.iloc[:train_size]
X_val = X.iloc[train_size:train_size + val_size]
X_test = X.iloc[train_size + val_size:]

y_train = y.iloc[:train_size]
y_val = y.iloc[train_size:train_size + val_size]
y_test = y.iloc[train_size + val_size:]

In [63]:
X_train

,PJME_MW,Hour,Day,Week,Month,DayOfWeek,IsWeekend,Lag_1,Lag_24,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24
0,27213.0,1,24,52,12,3,0,28570.0,27669.0,26498.0,29475.375000,30894.589286,2736.646326
1,25643.0,2,24,52,12,3,0,27213.0,26162.0,25147.0,29453.750000,30897.541667,2765.861628
2,24907.0,3,24,52,12,3,0,25643.0,25483.0,24574.0,29429.750000,30899.523810,2804.050165
3,24721.0,4,24,52,12,3,0,24907.0,25045.0,24393.0,29416.250000,30901.476190,2826.766154
4,25144.0,5,24,52,12,3,0,24721.0,25030.0,24860.0,29421.000000,30903.166667,2819.160745
...,...,...,...,...,...,...,...,...,...,...,...,...,...
101633,41337.0,18,20,8,2,4,0,40650.0,43673.0,32288.0,33723.791667,28494.648810,6725.675004
101634,40576.0,19,20,8,2,4,0,41337.0,42547.0,31967.0,33641.666667,28545.892857,6624.526508
101635,39599.0,20,20,8,2,4,0,40576.0,41322.0,31616.0,33569.875000,28593.410714,6546.551321
101636,39816.0,21,20,8,2,4,0,39599.0,41054.0,31892.0,33518.291667,28640.577381,6489.646527


In [64]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)

X_val_scaled = X_scaler.transform(X_val)

X_test_scaled = X_scaler.transform(X_test)

In [65]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = y_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1, 1)
)

In [66]:
sequence_length = 168
forecast_horizon = 24

def create_sequences(X, y, sequence_length, forecast_horizon):

    X_sequences = []
    y_sequences = []

    for i in range(
        sequence_length,
        len(X) - forecast_horizon + 1
    ):

        X_sequences.append(
            X[i-sequence_length:i]
        )

        y_sequences.append(
            y[i:i+forecast_horizon]
        )

    return np.array(X_sequences), np.array(y_sequences)

In [ ]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [68]:
print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101447, 168, 13)
y_train: (101447, 24, 1)
X_val: (21588, 168, 13)
y_val: (21588, 24, 1)
X_test: (21590, 168, 13)
y_test: (21590, 24, 1)


# Best Model Bidirectional LSTM

In [69]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [70]:
bilstm_model = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=13
)

bilstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0987 - mae: 0.2259 - val_loss: 0.0872 - val_mae: 0.2230
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.0561 - mae: 0.1758 - val_loss: 0.0784 - val_mae: 0.2065
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0483 - mae: 0.1627 - val_loss: 0.0805 - val_mae: 0.2119
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0428 - mae: 0.1531 - val_loss: 0.0793 - val_mae: 0.2069
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0434 - mae: 0.1539 - val_loss: 0.1005 - val_mae: 0.2383
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0494 - mae: 0.1670 - val_loss: 0.1002 - val_mae: 0.2356
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0390 - mae: 0.1480 - val_loss: 0.0952 - val_mae: 0.2276
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0338 - mae: 0.1377 - val_loss: 0.0957 - val_mae: 0.2277
Epoch 9/10
1586/1586 ━━━

# RNN 168

In [71]:
def build_rnn_model(sequence_length, n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(24)
    ])

    return model

In [72]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=13
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 25s 15ms/step - loss: 0.1341 - mae: 0.2669 - val_loss: 0.1280 - val_mae: 0.2679
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0779 - mae: 0.2079 - val_loss: 0.1193 - val_mae: 0.2581
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0695 - mae: 0.1953 - val_loss: 0.1424 - val_mae: 0.2856
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0654 - mae: 0.1888 - val_loss: 0.1377 - val_mae: 0.2782
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0621 - mae: 0.1837 - val_loss: 0.1056 - val_mae: 0.2430
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0594 - mae: 0.1795 - val_loss: 0.1059 - val_mae: 0.2426
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0569 - mae: 0.1754 - val_loss: 0.1311 - val_mae: 0.2699
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0547 - mae: 0.1722 - val_loss: 0.1288 - val_mae: 0.2673
Epoch 9/10
1586/1586 ━━━

# GRU

In [73]:
def build_gru_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.GRU(64),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [74]:
gru_model = build_gru_model(
    sequence_length=sequence_length,
    n_features=13
)

gru_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = gru_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1261 - mae: 0.2569 - val_loss: 0.1403 - val_mae: 0.2825
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0651 - mae: 0.1892 - val_loss: 0.1036 - val_mae: 0.2438
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0574 - mae: 0.1764 - val_loss: 0.1213 - val_mae: 0.2649
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0527 - mae: 0.1687 - val_loss: 0.1230 - val_mae: 0.2644
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0492 - mae: 0.1628 - val_loss: 0.1224 - val_mae: 0.2557
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0460 - mae: 0.1572 - val_loss: 0.1237 - val_mae: 0.2524
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0431 - mae: 0.1522 - val_loss: 0.1185 - val_mae: 0.2462
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0407 - mae: 0.1479 - val_loss: 0.1272 - val_mae: 0.2561
Epoch 9/10
1586/1586 ━━━━━━━━━━━

# LSTM

In [75]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [76]:
lstm_model = build_lstm_model(
    sequence_length=sequence_length,
    n_features=13
)

lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1170 - mae: 0.2465 - val_loss: 0.1290 - val_mae: 0.2760
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0629 - mae: 0.1863 - val_loss: 0.0972 - val_mae: 0.2367
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0556 - mae: 0.1741 - val_loss: 0.1084 - val_mae: 0.2474
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0503 - mae: 0.1649 - val_loss: 0.1004 - val_mae: 0.2365
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0467 - mae: 0.1589 - val_loss: 0.1005 - val_mae: 0.2364
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0432 - mae: 0.1529 - val_loss: 0.1133 - val_mae: 0.2495
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0411 - mae: 0.1492 - val_loss: 0.1225 - val_mae: 0.2618
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0383 - mae: 0.1446 - val_loss: 0.1080 - val_mae: 0.2444
Epoch 9/10
1586/1586 ━━━━━━━━━━

In [77]:
rnn_pred = rnn_model.predict(
    X_test_seq
)

lstm_pred = lstm_model.predict(
    X_test_seq
)

gru_pred = gru_model.predict(
    X_test_seq
)

bilstm_pred = bilstm_model.predict(
    X_test_seq
)

675/675 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
675/675 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
675/675 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step


In [78]:
rnn_pred_original = y_scaler.inverse_transform(
    rnn_pred.reshape(-1, 1)
).reshape(rnn_pred.shape)

lstm_pred_original = y_scaler.inverse_transform(
    lstm_pred.reshape(-1, 1)
).reshape(lstm_pred.shape)

gru_pred_original = y_scaler.inverse_transform(
    gru_pred.reshape(-1, 1)
).reshape(gru_pred.shape)

bilstm_pred_original = y_scaler.inverse_transform(
    bilstm_pred.reshape(-1, 1)
).reshape(bilstm_pred.shape)

In [79]:
y_test_original = y_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [80]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [81]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original ,
    rnn_pred_original
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original ,
    lstm_pred_original
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original ,
    gru_pred_original
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original
)

In [82]:
deep_results

{'RNN': {'MAE': 2505.472982939331,
  'MSE': 11760047.902376242,
  'RMSE': np.float64(3429.292624197626),
  'MAPE': 8.372387724229403,
  'R2': 0.7187808440602956,
  'Bias': np.float64(1325.3893312544628)},
 'LSTM': {'MAE': 3084.0394673165924,
  'MSE': 16838518.85785036,
  'RMSE': np.float64(4103.476435639708),
  'MAPE': 10.450901804720294,
  'R2': 0.5973388799272956,
  'Bias': np.float64(1673.6073415012074)},
 'GRU': {'MAE': 3645.5560973150486,
  'MSE': 23684668.023592245,
  'RMSE': np.float64(4866.689637072847),
  'MAPE': 12.347271807942382,
  'R2': 0.4336262568317535,
  'Bias': np.float64(2286.382240543009)},
 'Bi-LSTM': {'MAE': 2600.279045915626,
  'MSE': 12048502.912268538,
  'RMSE': np.float64(3471.0953476199034),
  'MAPE': 8.817906254885974,
  'R2': 0.7118829916806212,
  'Bias': np.float64(1223.3554688743884)}}

In [83]:
deep_results_df_168 = pd.DataFrame(
    deep_results
).T

deep_results_df_168

,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,2505.472983,1.176005e+07,3429.292624,8.372388,0.718781,1325.389331
LSTM,3084.039467,1.683852e+07,4103.476436,10.450902,0.597339,1673.607342
GRU,3645.556097,2.368467e+07,4866.689637,12.347272,0.433626,2286.382241
Bi-LSTM,2600.279046,1.204850e+07,3471.095348,8.817906,0.711883,1223.355469


# 48

In [84]:
sequence_length = 48
forecast_horizon = 24

X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

# BILSTM 48

In [85]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=13
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1055 - mae: 0.2339 - val_loss: 0.0970 - val_mae: 0.2341
Epoch 2/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0590 - mae: 0.1798 - val_loss: 0.0845 - val_mae: 0.2158
Epoch 3/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0509 - mae: 0.1661 - val_loss: 0.0809 - val_mae: 0.2089
Epoch 4/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0456 - mae: 0.1570 - val_loss: 0.0919 - val_mae: 0.2299
Epoch 5/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0415 - mae: 0.1498 - val_loss: 0.0848 - val_mae: 0.2154
Epoch 6/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0379 - mae: 0.1435 - val_loss: 0.0868 - val_mae: 0.2165
Epoch 7/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0349 - mae: 0.1381 - val_loss: 0.0848 - val_mae: 0.2127
Epoch 8/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0324 - mae: 0.1336 - val_loss: 0.0923 - val_mae: 0.2223
Epoch 9/10
1587/1587 ━━━━━━━━━━━

# RNN 48

In [86]:
rnn_model_48 = build_rnn_model(
    sequence_length=sequence_length,
    n_features=13
)

rnn_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.1407 - mae: 0.2734 - val_loss: 0.1682 - val_mae: 0.3086
Epoch 2/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0811 - mae: 0.2121 - val_loss: 0.1493 - val_mae: 0.2858
Epoch 3/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0717 - mae: 0.1982 - val_loss: 0.1436 - val_mae: 0.2821
Epoch 4/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0670 - mae: 0.1910 - val_loss: 0.1393 - val_mae: 0.2791
Epoch 5/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0638 - mae: 0.1859 - val_loss: 0.1480 - val_mae: 0.2843
Epoch 6/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0611 - mae: 0.1819 - val_loss: 0.1482 - val_mae: 0.2846
Epoch 7/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0589 - mae: 0.1785 - val_loss: 0.1256 - val_mae: 0.2581
Epoch 8/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0574 - mae: 0.1763 - val_loss: 0.1349 - val_mae: 0.2690
Epoch 9/10
1587/1587 ━━━━━━━━━━━━━━━━━━

# GRU 48

In [87]:
gru_model_48 = build_gru_model(
    sequence_length=sequence_length,
    n_features=13
)

gru_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = gru_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.1257 - mae: 0.2574 - val_loss: 0.1495 - val_mae: 0.2874
Epoch 2/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0665 - mae: 0.1914 - val_loss: 0.1544 - val_mae: 0.2931
Epoch 3/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0592 - mae: 0.1794 - val_loss: 0.1251 - val_mae: 0.2608
Epoch 4/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0542 - mae: 0.1710 - val_loss: 0.1313 - val_mae: 0.2662
Epoch 5/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0502 - mae: 0.1640 - val_loss: 0.1198 - val_mae: 0.2502
Epoch 6/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0468 - mae: 0.1583 - val_loss: 0.1485 - val_mae: 0.2786
Epoch 7/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0440 - mae: 0.1533 - val_loss: 0.1346 - val_mae: 0.2598
Epoch 8/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0420 - mae: 0.1498 - val_loss: 0.1221 - val_mae: 0.2480
Epoch 9/10
1587/1587 ━━━━━━━━━━━━

# LSTM 48

In [88]:
lstm_model_48 = build_lstm_model(
    sequence_length=sequence_length,
    n_features=13
)

lstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - loss: 0.1173 - mae: 0.2463 - val_loss: 0.1443 - val_mae: 0.2883
Epoch 2/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0640 - mae: 0.1879 - val_loss: 0.1073 - val_mae: 0.2419
Epoch 3/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0573 - mae: 0.1766 - val_loss: 0.1297 - val_mae: 0.2678
Epoch 4/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0529 - mae: 0.1690 - val_loss: 0.1189 - val_mae: 0.2548
Epoch 5/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0492 - mae: 0.1627 - val_loss: 0.1168 - val_mae: 0.2539
Epoch 6/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0461 - mae: 0.1574 - val_loss: 0.1205 - val_mae: 0.2548
Epoch 7/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0432 - mae: 0.1525 - val_loss: 0.1299 - val_mae: 0.2632
Epoch 8/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0405 - mae: 0.1478 - val_loss: 0.1288 - val_mae: 0.2651
Epoch 9/10
1587/1587 ━━━━━━━━━━━

In [89]:
rnn_pred_48 = rnn_model_48.predict(
    X_test_seq
)

lstm_pred_48 = lstm_model_48.predict(
    X_test_seq
)

gru_pred_48 = gru_model_48.predict(
    X_test_seq
)

bilstm_pred_48 = bilstm_model_48.predict(
    X_test_seq
)

679/679 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
679/679 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
679/679 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
679/679 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [90]:
rnn_pred_original_48 = y_scaler.inverse_transform(
    rnn_pred_48.reshape(-1, 1)
).reshape(rnn_pred_48.shape)

lstm_pred_original_48 = y_scaler.inverse_transform(
    lstm_pred_48.reshape(-1, 1)
).reshape(lstm_pred_48.shape)

gru_pred_original_48 = y_scaler.inverse_transform(
    gru_pred_48.reshape(-1, 1)
).reshape(gru_pred_48.shape)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48.reshape(-1, 1)
).reshape(bilstm_pred_48.shape)

In [91]:
y_test_original = y_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [92]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [93]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_48
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_48
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original_48
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

In [94]:
deep_results

{'RNN': {'MAE': 3113.5024672822324,
  'MSE': 17937065.525450516,
  'RMSE': np.float64(4235.21729377024),
  'MAPE': 10.389930036694516,
  'R2': 0.569920373719578,
  'Bias': np.float64(951.4075939279794)},
 'LSTM': {'MAE': 3116.8659855731744,
  'MSE': 17748782.625118684,
  'RMSE': np.float64(4212.930408292865),
  'MAPE': 10.461481041970803,
  'R2': 0.5744348601775113,
  'Bias': np.float64(1788.955435309519)},
 'GRU': {'MAE': 3431.3661301982875,
  'MSE': 20900880.72662039,
  'RMSE': np.float64(4571.748104021086),
  'MAPE': 11.59890091786701,
  'R2': 0.4988565460118204,
  'Bias': np.float64(1739.27571381051)},
 'Bi-LSTM': {'MAE': 2483.9147090434517,
  'MSE': 11166770.968352918,
  'RMSE': np.float64(3341.6718822099992),
  'MAPE': 8.348158054753629,
  'R2': 0.7322527100091158,
  'Bias': np.float64(1040.4536914152463)}}

In [95]:
deep_results_df_48 = pd.DataFrame(
    deep_results
).T

deep_results_df_48

,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,3113.502467,1.793707e+07,4235.217294,10.389930,0.569920,951.407594
LSTM,3116.865986,1.774878e+07,4212.930408,10.461481,0.574435,1788.955435
GRU,3431.366130,2.090088e+07,4571.748104,11.598901,0.498857,1739.275714
Bi-LSTM,2483.914709,1.116677e+07,3341.671882,8.348158,0.732253,1040.453691


In [96]:
print("y_test_seq:", y_test_seq.shape)

print("RNN:", rnn_pred.shape)
print("LSTM:", lstm_pred.shape)
print("GRU:", gru_pred.shape)
print("Bi-LSTM:", bilstm_pred.shape)

print("y_test original:")
print(y_test_scaled[:2])

print("RNN original:")
print(rnn_pred_original[:2])

y_test_seq: (21710, 24, 1)
RNN: (21590, 24)
LSTM: (21590, 24)
GRU: (21590, 24)
Bi-LSTM: (21590, 24)
y_test original:
[[-0.43576355]
 [-0.40969518]]
RNN original:
[[28764.473 28031.654 27273.617 26647.656 26802.475 28073.549 29979.896
  31102.49  31169.266 30490.686 29223.113 27429.52  25559.207 24057.096
  22989.154 22728.746 23077.459 24595.04  27174.055 29921.484 31646.85
  31890.213 31757.178 31515.062]
 [28757.06  28393.332 28126.295 28152.684 29067.787 30530.982 31703.424
  31803.598 31174.957 29788.928 27847.701 25497.223 23842.465 22802.14
  22324.133 22670.318 24267.873 27149.867 30326.203 32276.344 32575.652
  32061.957 31659.607 31265.104]]


# 24

In [97]:
sequence_length = 24
forecast_horizon = 24

X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

# Bi_LSTM 24

In [98]:
bilstm_model_24 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=13
)

bilstm_model_24.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_24.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.1101 - mae: 0.2406 - val_loss: 0.1053 - val_mae: 0.2460
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0644 - mae: 0.1871 - val_loss: 0.1003 - val_mae: 0.2355
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0563 - mae: 0.1740 - val_loss: 0.1083 - val_mae: 0.2487
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0515 - mae: 0.1661 - val_loss: 0.1020 - val_mae: 0.2407
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0477 - mae: 0.1596 - val_loss: 0.0959 - val_mae: 0.2287
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0444 - mae: 0.1541 - val_loss: 0.1032 - val_mae: 0.2377
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0413 - mae: 0.1488 - val_loss: 0.1009 - val_mae: 0.2346
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0387 - mae: 0.1445 - val_loss: 0.1043 - val_mae: 0.2393
Epoch 9/10
1588/1588 ━━━━━━━━━━━

# RNN 24

In [99]:
rnn_model_24 = build_rnn_model(
    sequence_length=sequence_length,
    n_features=13
)

rnn_model_24.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_24.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.1370 - mae: 0.2704 - val_loss: 0.1583 - val_mae: 0.3016
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0820 - mae: 0.2127 - val_loss: 0.1562 - val_mae: 0.2994
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0715 - mae: 0.1976 - val_loss: 0.1297 - val_mae: 0.2683
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0670 - mae: 0.1905 - val_loss: 0.1182 - val_mae: 0.2587
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0640 - mae: 0.1855 - val_loss: 0.1105 - val_mae: 0.2482
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0623 - mae: 0.1824 - val_loss: 0.1168 - val_mae: 0.2539
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0607 - mae: 0.1800 - val_loss: 0.1010 - val_mae: 0.2359
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0594 - mae: 0.1777 - val_loss: 0.1220 - val_mae: 0.2630
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━

# GRU 24

In [100]:
gru_model_24 = build_gru_model(
    sequence_length=sequence_length,
    n_features=13
)

gru_model_24.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = gru_model_24.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.1302 - mae: 0.2618 - val_loss: 0.1592 - val_mae: 0.3030
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0712 - mae: 0.1978 - val_loss: 0.1480 - val_mae: 0.2878
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0638 - mae: 0.1858 - val_loss: 0.1318 - val_mae: 0.2725
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0595 - mae: 0.1785 - val_loss: 0.1366 - val_mae: 0.2768
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0564 - mae: 0.1733 - val_loss: 0.1428 - val_mae: 0.2826
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0539 - mae: 0.1690 - val_loss: 0.1460 - val_mae: 0.2862
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0517 - mae: 0.1653 - val_loss: 0.1437 - val_mae: 0.2804
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0498 - mae: 0.1620 - val_loss: 0.1561 - val_mae: 0.2919
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━

# LSTM 24

In [101]:
lstm_model_24 = build_lstm_model(
    sequence_length=sequence_length,
    n_features=13
)

lstm_model_24.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_24.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.1248 - mae: 0.2573 - val_loss: 0.1436 - val_mae: 0.2893
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0698 - mae: 0.1963 - val_loss: 0.1295 - val_mae: 0.2748
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0617 - mae: 0.1830 - val_loss: 0.1096 - val_mae: 0.2505
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0572 - mae: 0.1756 - val_loss: 0.1058 - val_mae: 0.2432
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0538 - mae: 0.1697 - val_loss: 0.1117 - val_mae: 0.2520
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0511 - mae: 0.1649 - val_loss: 0.1016 - val_mae: 0.2377
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0489 - mae: 0.1611 - val_loss: 0.1176 - val_mae: 0.2557
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0471 - mae: 0.1582 - val_loss: 0.1227 - val_mae: 0.2634
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━

In [102]:
rnn_pred_24 = rnn_model_24.predict(
    X_test_seq
)

lstm_pred_24 = lstm_model_24.predict(
    X_test_seq
)

gru_pred_24 = gru_model_24.predict(
    X_test_seq
)

bilstm_pred_24 = bilstm_model_24.predict(
    X_test_seq
)

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [104]:
rnn_pred_original_24 = y_scaler.inverse_transform(
    rnn_pred_24.reshape(-1, 1)
).reshape(rnn_pred_24.shape)

lstm_pred_original_24 = y_scaler.inverse_transform(
    lstm_pred_24.reshape(-1, 1)
).reshape(lstm_pred_24.shape)

gru_pred_original_24 = y_scaler.inverse_transform(
    gru_pred_24.reshape(-1, 1)
).reshape(gru_pred_24.shape)

bilstm_pred_original_24 = y_scaler.inverse_transform(
    bilstm_pred_24.reshape(-1, 1)
).reshape(bilstm_pred_24.shape)

In [105]:
y_test_original = y_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [106]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [107]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_24
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_24
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original_24
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_24
)

In [108]:
deep_results

{'RNN': {'MAE': 2846.4260242598243,
  'MSE': 14315979.704751419,
  'RMSE': np.float64(3783.6463503809946),
  'MAPE': 9.475379067244374,
  'R2': 0.6564371499827961,
  'Bias': np.float64(947.4050203478229)},
 'LSTM': {'MAE': 3283.3430272348637,
  'MSE': 18761523.218131285,
  'RMSE': np.float64(4331.457401167797),
  'MAPE': 11.142470505980006,
  'R2': 0.5497505221144032,
  'Bias': np.float64(2062.527912883502)},
 'GRU': {'MAE': 4117.805743956193,
  'MSE': 26966376.539645314,
  'RMSE': np.float64(5192.915995820202),
  'MAPE': 14.127954608369112,
  'R2': 0.35284588483156976,
  'Bias': np.float64(2411.6061911706543)},
 'Bi-LSTM': {'MAE': 2695.5888713063105,
  'MSE': 13269242.898125751,
  'RMSE': np.float64(3642.6972009934825),
  'MAPE': 8.972691639365236,
  'R2': 0.6815573225395415,
  'Bias': np.float64(1326.5356209491872)}}

In [109]:
deep_results_df_24 = pd.DataFrame(
    deep_results
).T

deep_results_df_24

,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,2846.426024,1.431598e+07,3783.646350,9.475379,0.656437,947.405020
LSTM,3283.343027,1.876152e+07,4331.457401,11.142471,0.549751,2062.527913
GRU,4117.805744,2.696638e+07,5192.915996,14.127955,0.352846,2411.606191
Bi-LSTM,2695.588871,1.326924e+07,3642.697201,8.972692,0.681557,1326.535621


In [110]:
bilstm_model_48.save('/kaggle/working/Base_best_model.keras')